# Titanic - Survival Prediction

This project explores the Titanic dataset available on Kaggle.
The goal is to understand which factors influenced passenger survival during the Titanic disaster.

## 1. Problem Understanding

The Titanic dataset contains information about passengers such as age, gender, passenger class, and survival status.

The main question addressed in this project is:

**What factors influenced survival on the Titanic?**

From a machine learning perspective, this is a **binary classification problem**, where:
- `1` represents passengers who survived
- `0` represents passengers who did not survive

The objective is to build a model capable of predicting survival based on passenger characteristics.


## 2. Import Libraries

The following Python libraries are used for data analysis, visualization, and machine learning.
Only essential libraries are imported to keep the notebook simple and easy to understand.


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

## 3. Load the Dataset

The Titanic competition provides:
- `train.csv` for training (includes the target column `Survived`)
- `test.csv` for evaluation/submission (does not include `Survived`)


In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

train_df.head()

## 4. Data Overview

The dataset structure is inspected before any preprocessing.
This helps identify:
- number of rows and columns
- data types
- missing values
- the target variable

In [ ]:
train_df.shape

In [ ]:
train_df.info()

From the dataset information, the following observations can be made:

- Each row represents a single passenger.
- The dataset contains both numerical and categorical variables.
- Some variables contain missing values that must be handled before modeling.

The target variable of this project is **Survived**, which indicates whether a passenger survived the disaster.

## 5. Missing Values

Most machine learning models cannot train correctly with missing values.
In this section:
- columns with missing values are identified
- simple and defensible strategies are applied


In [ ]:
train_df.isna().sum().sort_values(ascending=False)

In [ ]:
test_df.isna().sum().sort_values(ascending=False)

Handling strategy:
- **Cabin** has too many missing values and is removed to avoid complex assumptions.
- **Age** is filled with the median value (a robust “typical” value).
- **Embarked** is filled with the most common value (mode).
- **Fare** in the test set contains a small number of missing values and is filled with the median fare.

In [ ]:
# Drop Cabin (too many missing values)
train_df = train_df.drop(columns=["Cabin"])
test_df  = test_df.drop(columns=["Cabin"])

# Fill Age with median (computed from training set)
age_median = train_df["Age"].median()
train_df["Age"] = train_df["Age"].fillna(age_median)
test_df["Age"]  = test_df["Age"].fillna(age_median)

# Fill Embarked with mode (computed from training set)
emb_mode = train_df["Embarked"].mode()[0]
train_df["Embarked"] = train_df["Embarked"].fillna(emb_mode)
test_df["Embarked"]  = test_df["Embarked"].fillna(emb_mode)

# Fill Fare missing values in test set (use training median for consistency)
fare_median = train_df["Fare"].median()
test_df["Fare"] = test_df["Fare"].fillna(fare_median)

In [ ]:
train_df[["Age", "Embarked"]].isna().sum()

In [ ]:
test_df[["Age", "Embarked", "Fare"]].isna().sum()

## 6. Remove non-informative identifiers
Columns like `Name` and `Ticket` are identifiers rather than explanatory passenger characteristics.
They are removed to reduce noise and improve interpretability.

In [ ]:
train_df = train_df.drop(columns=["Name", "Ticket"])
train_df.head()

## 7. Correlation Heatmap
Before creating many engineered variables, we check the **numeric** relationships.

Why:
- to avoid redundant variables
- to justify dropping features when appropriate

We focus on numeric variables only, since correlation is defined for numeric data.

In [ ]:
numeric_cols = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]
corr_matrix = train_df[numeric_cols].corr()

plt.figure(figsize=(7,6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap (Numeric Variables)")
plt.show()

corr_matrix

### Decision: drop Fare 
The heatmap shows that `Fare` and `Pclass` are **moderately correlated** (≈ -0.55), meaning both capture overlapping socioeconomic information.

For clarity and to reduce redundancy (especially for reporting), we:
- keep `Pclass` (more interpretable)
- drop `Fare`

In [ ]:
train_df = train_df.drop(columns=["Fare"])
test_df = test_df.drop(columns=["Fare"])

## 8. Feature Engineering

Feature engineering creates new variables from existing ones.
Only simple and intuitive features are added:

### 8.1 Family Features

- **FamilySize** = SibSp + Parch + 1  
  This represents the total number of family members traveling together (including the passenger).

- **IsAlone** = 1 if SibSp + Parch == 0, else 0  
  This indicates whether the passenger was traveling alone.


In [ ]:
def add_family_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = np.where((df["SibSp"] + df["Parch"]) == 0, 1, 0)
    return df

train_df = add_family_features(train_df)
test_df  = add_family_features(test_df)

train_df[["SibSp","Parch","FamilySize","IsAlone"]].head()


### 8.2 Age Categorization

Age is converted into categorical groups based on commonly used demographic age ranges:
- Child (0–14)
- Young (15–24)
- Adult (25–64)
- Senior (65+)

In [ ]:
def age_to_group(age: float) -> str:
    if age <= 14:
        return "Child"
    elif age <= 24:
        return "Young"
    elif age <= 64:
        return "Adult"
    else:
        return "Senior"

train_df["AgeGroup"] = train_df["Age"].apply(age_to_group)
test_df["AgeGroup"]  = test_df["Age"].apply(age_to_group)

train_df[["Age", "AgeGroup"]].head()


## 9. Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) is used to observe patterns and build intuition about which variables relate to passenger survival. The objective is to understand how survival varies across different passenger groups and to validate historical assumptions using data.

The following questions are explored:

- Does survival differ by gender?
- Does survival differ by passenger class?
- Does survival vary across predefined age groups?
- Do passengers traveling alone have different survival outcomes?

In [ ]:
sns.barplot(x="Sex", y="Survived", data=train_df)
plt.title("Survival Rate by Gender")
plt.ylabel("Survival Rate")
plt.show()

Female passengers show a higher survival rate than male passengers.

In [ ]:
sns.barplot(x="Pclass", y="Survived", data=train_df)
plt.title("Survival Rate by Passenger Class")
plt.ylabel("Survival Rate")
plt.show()

Passengers in 1st class show higher survival rates than passengers in lower classes.

In [ ]:
sns.barplot(x="AgeGroup", y="Survived", data=train_df, order=["Child","Young","Adult","Senior"])
plt.title("Survival Rate by Age Group")
plt.ylabel("Survival Rate")
plt.show()

Survival rates vary across predefined age groups. Childs are more likely to survive.

In [ ]:
sns.barplot(x="IsAlone", y="Survived", data=train_df)
plt.title("Survival Rate: Traveling Alone vs Not Alone")
plt.ylabel("Survival Rate")
plt.show()

Survival appears to differ between passengers traveling alone and those traveling with family.

## 10. Data Preparation

Machine learning models require numerical input.
This section prepares the data by:
- selecting a set of relevant features
- converting categorical variables to numeric (one-hot encoding)
- splitting features (X) and target (y)

### 10.1 Feature Selection

The selected features are chosen because they are:

- interpretable and easy to explain
- consistent with the exploratory analysis
- available in both training and test datasets

Numerical variable that was not normalized (Age) was converted into categorical variable (AgeGroup) to ensure consistency and interpretability.

In [ ]:
features = [
    "Pclass",
    "Sex",
    "AgeGroup",
    "FamilySize",
    "IsAlone",
    "Embarked"
]

X = train_df[features].copy()
y = train_df["Survived"].copy()
X_test = test_df[features].copy()


### 10.2 Encoding Categorical Variables

Categorical variables (Sex, Embarked, AgeGroup) must be converted into numeric columns.
One-hot encoding is used because it:
- does not impose an artificial order between categories
- is widely used and easy to interpret


In [ ]:
# Encode TRAIN
X_encoded = pd.get_dummies(
    X,
    columns=["Sex", "Embarked", "AgeGroup"],
    drop_first=True
).astype(int)

# Encode TEST
X_test_encoded = pd.get_dummies(
    X_test,
    columns=["Sex", "Embarked", "AgeGroup"],
    drop_first=True
).astype(int)

# Align columns
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

X_encoded.shape, X_test_encoded.shape

### 10.3 Train-Test Split

A train-test split is used to evaluate model performance on unseen data.
This helps estimate how well the model generalizes beyond the training set.

80% of the data is used for training and 20% for testing.
A fixed random seed is used for reproducibility.


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

## 11. Model Selection 

Only two models are tested to keep the comparison meaningful and easy to explain.

### Logistic Regression 
Logistic Regression is a standard baseline for binary classification.
It is chosen because:
- it is simple and widely used,
- it supports interpretability (feature effects can be discussed more directly),
- it provides a strong reference point for performance comparisons.

### Random Forest
Random Forest is included to compare performance with a more powerful model.
It is chosen because:
- it captures non-linear relationships,
- it usually performs well on tabular datasets,
- it provides a realistic example of how a more complex model can improve accuracy.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42)
}

## 12. Evaluation Metric 

Accuracy is used as the primary evaluation metric in this project.

**Accuracy measures the proportion of correct predictions**:
- (number of correct predictions) / (total predictions)

Accuracy is appropriate here because:
- the classes (survived vs not survived) are reasonably balanced,
- the objective is an overall measure of correct classification,
- it is straightforward to interpret.

Other metrics such as precision, recall, and F1-score become more critical when:
- the dataset is highly imbalanced, or
- false positives and false negatives have very different costs.

## 13. Cross-Validation
A single train/validation split can be sensitive to how data is split.
Cross-validation reduces this sensitivity by evaluating the model across multiple folds.

This section reports:
- Mean accuracy across folds
- Standard deviation (stability across folds)

In [ ]:
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_results = []
for name, model in models.items():
    scores = cross_val_score(model, X_encoded, y, cv=kfold, scoring="accuracy")
    cv_results.append({
        "Model": name,
        "Mean Accuracy": scores.mean(),
        "Std Dev": scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values(by="Mean Accuracy", ascending=False)
cv_df

## 14. Hold-Out Validation

Models are trained on the training split and evaluated on the validation split.
This provides an intuitive comparison using unseen data.


In [ ]:
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    acc = accuracy_score(y_valid, preds)
    trained_models[name] = (model, acc)

val_df = pd.DataFrame([
    {"Model": k, "Validation Accuracy": v[1]}
    for k, v in trained_models.items()
]).sort_values(by="Validation Accuracy", ascending=False)

val_df

## 15. Hyperparameter Tuning (Random Forest)

Hyperparameters are settings chosen **before training** (not learned from data),
such as the number of trees or the maximum depth of each tree.

Random Forest performance can depend significantly on these settings.
To improve performance in a controlled way, GridSearchCV is used to test combinations
of hyperparameters using cross-validation.

The tuned model is then compared against:
- Logistic Regression
- Random Forest baseline

In [ ]:
rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 4, 6, 8],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="accuracy",
    cv=kfold,
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_rf = grid.best_estimator_
grid.best_params_, grid.best_score_


## 16. Tuned Random Forest – Hold-out Validation

In [ ]:
best_rf.fit(X_train, y_train)
rf_tuned_preds = best_rf.predict(X_valid)

rf_tuned_acc = accuracy_score(y_valid, rf_tuned_preds)
rf_tuned_acc

## 17. Model Comparison Summary

To evaluate the impact of hyperparameter tuning, three models are compared:

- Logistic Regression (baseline, interpretable)
- Random Forest (baseline, default parameters)
- Random Forest (tuned using GridSearchCV)

This comparison allows assessing whether hyperparameter tuning leads to a measurable improvement in validation performance.


In [ ]:
summary_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Validation Accuracy": trained_models["Logistic Regression"][1]
    },
    {
        "Model": "Random Forest (baseline)",
        "Validation Accuracy": trained_models["Random Forest"][1]
    },
    {
        "Model": "Random Forest (tuned)",
        "Validation Accuracy": rf_tuned_acc
    }
]).sort_values(by="Validation Accuracy", ascending=False)

summary_df


## 18. Logistic Regression – Model Coefficients

Logistic Regression provides a linear and interpretable model.  
Each coefficient represents the direction and relative influence of a feature
on the probability of survival.

- **Positive coefficients** increase the likelihood of survival
- **Negative coefficients** decrease the likelihood of survival
- The magnitude of the coefficient indicates the strength of the effect,
  assuming all other variables are held constant

Because features were encoded using one-hot encoding, each coefficient
represents the effect of belonging to that category compared to the reference
category.

In [ ]:
lr_model = models["Logistic Regression"]

coef_df = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": lr_model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

coef_df

From the coefficients, it is possible to observe that:

- Gender-related variables have the strongest impact on survival probability
- Passenger class and age-related features also play an important role
- Family-related features have a smaller influence compared to gender and class

These results are consistent with historical accounts of the Titanic disaster,
where women, children, and higher-class passengers had higher survival rates.

## 19. Random Forest – Feature Importance

Unlike Logistic Regression, Random Forest is a non-linear ensemble model.
It does not provide coefficients, but instead measures **feature importance**.

Feature importance represents how much each variable contributes to reducing
impurity across all trees in the forest.

Higher importance values indicate that a feature is more influential in the
model’s decision-making process.

In [ ]:
rf_importance_df = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Importance": best_rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

rf_importance_df

The feature importance analysis shows that:

- Gender (Sex_male) is the most influential variable, followed by passenger class.
- Age-related and family-related features also contribute to predictions

This confirms that different model families identify similar key factors, reinforcing the robustness of the findings.


## 20. Final interpretation
Main survival factors (consistent across EDA + model interpretation):
- **Gender** (female passengers show higher survival)
- **Passenger class** (1st class higher survival)
- **Age group** (children higher survival)
- **Family context** (alone vs not alone differs)
- **Embarked location** shows a smaller effect compared to the factors above.

## 21. Export processed data
We export feature-engineered datasets suitable for visualization and analysis.

In [ ]:
train_export = train_df.copy()
test_export = test_df.copy()

train_export.to_csv("titanic_train_processed.csv", index=False)
test_export.to_csv("titanic_test_processed.csv", index=False)

train_export.head()
test_export.head()

## 22. Kaggle submission

Creates the submission.csv

In [ ]:
# Train the final Random Forest baseline using the full training set
rf_final = RandomForestClassifier(random_state=42)
rf_final.fit(X_encoded, y)

# Generate predictions for the Kaggle test set (using the same preprocessing / encoding as the training data)
test_preds = rf_final.predict(X_test_encoded)

# Create the submission.csv file
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": test_preds.astype(int)
})
submission.to_csv("submission.csv", index=False)

submission.head()
submission.shape
submission.columns